## Score Candidates

In [ ]:
import json
from concurrent.futures import Future, ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Literal

import pandas as pd
from IPython.display import Markdown, display
from langchain_anthropic import ChatAnthropic
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from recruit import Candidates, CandidatesDB

# Set global cache to in-memory
set_llm_cache(InMemoryCache())

In [ ]:
class RoleCriterion(BaseModel):
    name: str = Field(
        ...,
        description="Short label for the criterion (e.g., 'Master's in CS', '5+ Years Backend', 'React Proficiency')",
    )
    description: str = Field(
        ...,
        description="Concise description of the requirement covering skill, education, experience, or background",
    )


class RoleRequirements(BaseModel):
    required: list[RoleCriterion] = Field(
        ...,
        description="Mandatory, non-negotiable requirements for the role",
    )
    preferred: list[RoleCriterion] = Field(
        ...,
        description="Nice-to-have qualifications and preferred background",
    )

In [ ]:
# llm = ChatOpenAI(
#     model="gpt-5.6-sol",
#     temperature=0.0,
# )
llm = ChatAnthropic(model="claude-opus-5")  # type: ignore

In [ ]:
requirements_agent = llm.with_structured_output(
    RoleRequirements,
    method="json_schema",
)

In [ ]:
openenings = Path("..") / "openings"
# job_description_path = openenings / "frontend-engineer-berlin.md"
job_description_path = openenings / "technical-founders-associate-berlin.md"
# job_description_path = openenings / "ai-success-engineer-berlin.md"
# job_description_path = openenings / "president-and-coo-london-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

In [ ]:
system_prompt = f"""
Extract assessment criteria from the job description that can be objectively evaluated from a candidate's resume or profile.

Scope of Criteria:
- Technical Skills & Tools (languages, frameworks, platforms)
- Work Experience & Seniority (years in role, track record, leadership, company stage/scale)
- Education & Domain Background (degrees, fields of study, relevant industry experience)

Guidelines:
- Ensure all criteria are mutually exclusive and collectively cover the key aspects of the role.
- Focus strictly on verifiable attributes observable from a CV/profile (avoid generic soft skills like "curious" or "team player").
- Required: Non-negotiable, mandatory criteria (limit to 3 core criteria).
- Preferred: Strongly advantageous, nice-to-have qualifications (limit to 5-8 criteria).
""".strip()

In [ ]:
messages = [
    ("system", system_prompt),
    ("human", job_description),
]

raw = requirements_agent.invoke(messages)
requirements = RoleRequirements.model_validate(raw)

In [ ]:
print(requirements.model_dump_json(indent=2))

In [ ]:
class CriterionEvaluation(RoleCriterion):
    rating: Literal["strong", "moderate", "weak", "none", "unknown"] = Field(
        ...,
        description="Assessment of candidate fit against the criterion: strong, moderate, weak, none, or unknown",
    )
    evidence: str = Field(
        ...,
        description="Direct evidence or reasoning from the resume supporting the rating",
    )


class CandidateAssessment(BaseModel):
    required: list[CriterionEvaluation] = Field(
        ...,
        description="Evaluations against mandatory requirements",
    )
    preferred: list[CriterionEvaluation] = Field(
        ...,
        description="Evaluations against preferred qualifications",
    )

In [ ]:
def calculate_candidate_score(candidate: CandidateAssessment) -> int:
    """
    Calculate the overall score for the candidate based on required and nice-to-have skills.
    Required skills are weighted more heavily than nice-to-have skills.
    """

    score = 0

    for eval in candidate.required:
        match eval.rating:
            case "strong":
                score += 3 * 2
            case "moderate":
                score += 2 * 2
            case "weak":
                score += 1 * 2

    for eval in candidate.preferred:
        match eval.rating:
            case "strong":
                score += 3
            case "moderate":
                score += 2
            case "weak":
                score += 1
            case "none":
                score -= 3
            case _:
                score += 0

    return score

In [ ]:
def has_required_criteria(candidate: CandidateAssessment) -> bool:
    """
    Check if the candidate meets all required criteria.
    Returns True if all required criteria are rated as 'strong' or 'moderate'.
    """

    for eval in candidate.required:
        if eval.rating not in ["strong", "moderate"]:
            return False

    return True

In [ ]:
score_prompt = f"""
Judge the candidate's qualifications against the extracted requirements.

Requirements:
```json
{requirements.model_dump_json(indent=2)}
```
""".strip()

In [ ]:
llm = ChatAnthropic(model="claude-sonnet-5")  # type: ignore


def assess_candidate(resume: dict) -> CandidateAssessment:
    score_agent = llm.with_structured_output(
        CandidateAssessment,
        method="json_schema",
    )

    messages = [
        ("system", score_prompt),
        # ("human", job_description),
        # ("ai", requirements.model_dump_json(indent=2)),
        ("human", json.dumps(resume, indent=2)),
    ]

    raw = score_agent.invoke(messages)
    return CandidateAssessment.model_validate(raw)

In [ ]:
candidates = Candidates()
resume: dict = candidates.get_person(485761352) or {}
print(assess_candidate(resume).model_dump_json(indent=2))

In [ ]:
resume_scores = []
resumes_list = list(candidates)
# [40:80] + [
#     candidates.get_person(485761352) or {}
# ]  # Add a specific candidate for testing
resumes_scores: dict[int, CandidateAssessment] = {}

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {
        executor.submit(assess_candidate, resume): resume for resume in resumes_list
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Scoring candidates",
    ):
        try:
            score = future.result()
            resume = futures[future]
            resumes_scores[resume.get("id", -1)] = score

            profile_score = {
                "id": resume.get("id"),
                "full_name": resume.get("full_name"),
                "linkedin_url": resume.get("linkedin_url"),
                "score": calculate_candidate_score(score),
                "meets_required": has_required_criteria(score),
            }

            resume_scores.append(profile_score)

        except Exception as e:
            print(f"Error scoring candidate: {e}")

In [ ]:
resume_scores_df = pd.DataFrame(resume_scores)
resume_scores_df = resume_scores_df.sort_values(by="score", ascending=False)
resume_scores_df

In [ ]:
resume_scores_df[resume_scores_df["meets_required"]]

In [ ]:
print(resumes_scores[201793538].model_dump_json(indent=2))

In [ ]:
assessments_db = CandidatesDB("../spi/assessments")

for candidate_id, assessment in resumes_scores.items():
    if candidate_id == 335187441:
        print(f"Skipping candidate with invalid ID: {candidate_id}")
        continue

    resume = candidates.get_person(candidate_id) or {}
    print(f"Storing assessment for candidate {candidate_id}: {resume.get('full_name')}")
    assessments_db.store(
        {
            "id": candidate_id,
            "full_name": resume.get("full_name"),
            **assessment.model_dump(),
        }
    )